In [25]:
import pandas as pd
import numpy as np
import gc
import io
import os
from itertools import combinations
from tqdm import tqdm

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils
import modules.encode as encode

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

import lightgbm as lgb

print(lgb.__file__)

HEAD = 100000

SEED = 71

d:\Data Science\venv\Lib\site-packages\lightgbm\__init__.py


In [28]:
def handle_low_variance(df, variance_threshold=0.0001):
    variances = df.var()
    to_drop = variances[variances <= variance_threshold].index.tolist()
    filtered_df = df.drop(to_drop, axis=1)
    if to_drop:  # Kiểm tra xem to_drop có rỗng không trước khi in
        print(f"phương sai thấp: {to_drop[0]}...")  # In phần tử đầu tiên, ...
    else:
        print("Không có features nào có phương sai thấp hơn ngưỡng.")
    return filtered_df, to_drop

def read_feather_with_head(file_path):
    return pd.read_feather(file_path).head(HEAD)

def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + line.strip() + ".f" for line in f]
        return features

feature_paths = utils.get_feature_paths(prefixes=["f0", "f101"])
chunk_size = 10
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

# multi thread
with ThreadPoolExecutor(max_workers=10) as executor, \
    open(os.path.join(ROOT, "data/result/used_.txt"), "w") as f_selected, \
    open(os.path.join(ROOT, "data/result/unused_.txt"), "w") as f_unselected:
    
    selected_features_all = []
    unselected_features_all = []
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)

        X_filtered, unselected_features = handle_low_variance(X)
        selected_features = X_filtered.columns.tolist()
        
        selected_features_all.extend(selected_features)
        unselected_features_all.extend(unselected_features)
        
        print(len(selected_features), len(unselected_features))
        
    f_selected.write("\n".join(selected_features_all)) # đọc ghi file nên chỉ làm 1 lần vì bất đồng bộ tốn thời gian
    f_unselected.write("\n".join(unselected_features_all))
    print(len(selected_features) + len(unselected_features))

Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
Không có features nào có phương sai thấp hơn ngưỡng.
10 0
phương sai thấp: f001_FLAG_DOCUMENT_12...
8 2
phương sai thấp: f001_FLAG_DOCUMENT_2...
9 1
phương sai thấp: f001_FLAG_MOBIL...
9 1
Không có features nào có phương sai thấp hơn ngưỡng.
10 0

In [23]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count

def handle_low_correlation_with_target(df, target_column, threshold=0.02):
    correlations = df.corr()[target_column].abs().drop(target_column)
    low_corr_columns = correlations[correlations < threshold].index.tolist()
    return df.drop(low_corr_columns, axis=1), low_corr_columns

def read_feather_with_head(file_path):
    return pd.read_feather(file_path).head(HEAD)

def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + line.strip() + ".f" for line in f]
        return features

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

feature_paths = read(os.path.join(ROOT, "data/result/used.txt"))
chunk_size = 10
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

count = 0

# multi thread: 20 phút => 1 phút 26s
with ThreadPoolExecutor(max_workers=10) as executor, \
    open(os.path.join(ROOT, "data/result/used2.txt"), "w") as f_selected, \
    open(os.path.join(ROOT, "data/result/unused2.txt"), "w") as f_unselected:
    
    selected_features_all = []
    unselected_features_all = []
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat([pd.concat(chunk_dfs, axis=1), target], axis=1)

        X_filtered, unselected_features = handle_low_correlation_with_target(X, target_column="TARGET")
        selected_features = X_filtered.columns.tolist()
        selected_features.remove("TARGET")
        
        selected_features_all.extend(selected_features)
        unselected_features_all.extend(unselected_features)
        
        print(len(selected_features), len(unselected_features))
        
    f_selected.write("\n".join(selected_features_all)) # đọc ghi file nên chỉ làm 1 lần vì bất đồng bộ tốn thời gian
    f_unselected.write("\n".join(unselected_features_all))
    print(len(selected_features) + len(unselected_features))

d:\Data Science\venv\Lib\site-packages\lightgbm\__init__.py
4 6
8 2
2 8
4 6
4 6
2 8
1 9
3 7
1 9
4 6
1 9
5 5
5 5
9 1
0 10
3 7
9 1
3 7
2 8
9 1
3 7
6 4
7 3
6 4
5 5
3 7
1 9
2 8
0 10
0 10
0 10
0 10
0 10
3 7
3 7
0 10
3 7
0 10
0 10
0 10
0 10
2 8
1 9
0 10
0 10
2 8
0 10
2 8
0 10
0 10
1 9
1 9
0 10
0 10
2 8
1 9
2 8
1 9
0 10
1 9
0 10
0 10
0 10
1 9
2 8
2 8
1 9
4 6
3 7
1 9
1 9
0 10
1 9
1 9
0 10
1 9
6 4
5 5
0 10
0 10
0 10
0 10
0 10
0 10
0 10
0 10
0 10
0 10
0 10
2 8
1 9
4 6
3 7
3 7
1 9
2 8
2 8
2 8
2 8
2 8
1 9
0 10
2 8
1 9
2 8
2 8
1 9
0 10
2 8
1 9
2 8
1 9
0 10
1 9
0 10
0 10
0 10
1 9
2 8
1 9
1 9
5 5
5 5
2 8
4 6
7 3
4 6
7 3
3 7
2 8
2 8
6 4
6 4
4 6
8 2
0 10
1 9
5 5
10 0
8 2
9 1
7 3
0 10
0 10
1 9
7 3
5 5
6 4
6 4
4 6
0 10
0 10
3 7
3 7
5 5
6 4
6 4
0 10
0 10
0 10
5 5
4 6
0 10
0 10
0 10
0 10
0 10
0 10
0 10
2 8
0 10
0 10
0 10
6 4
0 10
0 10
8 2
5 5
0 10
0 10
1 9
3 7
7 3
3 7
0 10
0 10
2 8
3 7
4 6
3 7
4 6
1 9
2 8
1 9
1 9
5 5
3 7
1 9
3 7
2 8
1 9
3 7
4 6
5 5
4 6
2 8
3 7
2 8
1 9
2 8
0 10
1 9
1 9
0 10
0 10
1 9
2 8
0 1